In [59]:
import numpy as np
import plotly.express as px
import pandas as pd
from sklearn.metrics import confusion_matrix
import plotly.figure_factory as ff
from sklearn.linear_model import LogisticRegression as SklearnLR

# Logistic Regression

Logistic regression is mainly used for binary classification problems. Unlike linear regression which predicts a continuous value, the output here is a probability between 0 and 1, indicating the likelihood of belonging to a class.

---

## The Model

The formula is similar to linear regression with one key difference, the output is passed through a **sigmoid function** that squashes it between 0 and 1:

$$f_{\mathbf{w},b}(\mathbf{x}) = g(\mathbf{w} \cdot \mathbf{x} + b)$$

Where each variable represents:

- $f_{\mathbf{w},b}(\mathbf{x})$: The predicted probability
- $\mathbf{w}$: The weight vector, one weight per input feature
- $b$: The bias
- $\mathbf{x}$: The input feature vector
- $g$: The sigmoid function

---

## The Sigmoid Function

$$g(z) = \frac{1}{1 + e^{-z}}$$

Substituting $z = \mathbf{w} \cdot \mathbf{x} + b$ gives us the full model:

$$f_{\mathbf{w},b}(\mathbf{x}) = \frac{1}{1 + e^{-(\mathbf{w} \cdot \mathbf{x} + b)}}$$

In [63]:
z = np.linspace(-10,10,100)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

f = px.line(x=z, y=sigmoid(z), title="The Sigmoid function")
f.show()    
    

---

## The Cost Function

The cost function (which is used to measure how wrong our model is) is calculated using binary cross-entropy. A key property of this loss function is:

- **Asymmetric penalties:** The model is penalized heavily when it is confidently wrong (e.g. predicting $0.05$ when the true value is $1$)

$$J(\mathbf{w},b) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(f_{\mathbf{w},b}(\mathbf{x}^{(i)})) + (1 - y^{(i)}) \log(1 - f_{\mathbf{w},b}(\mathbf{x}^{(i)})) \right]$$

Where:

- $J(\mathbf{w},b)$: The cost
- $m$: The number of training examples
- $i$: Denotes which example we are on
- $\mathbf{x}^{(i)}$: The $i$ th input feature vector
- $y^{(i)}$: The $i$ th actual output value (either 0 or 1)
- $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$: The predicted probability for the $i$ th example

---

## Gradient Descent

Just like linear regression, we minimize the cost using gradient descent. The update rules are identical in form:

$$\mathbf{w} := \mathbf{w} - \alpha \frac{\partial J(\mathbf{w},b)}{\partial \mathbf{w}}$$

$$b := b - \alpha \frac{\partial J(\mathbf{w},b)}{\partial b}$$

Where the partial derivatives expand to:

$$\frac{\partial J(\mathbf{w},b)}{\partial \mathbf{w}} = \frac{1}{m} \sum_{i=1}^{m} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}) \mathbf{x}^{(i)}$$

$$\frac{\partial J(\mathbf{w},b)}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})$$

The gradient equations look identical to those in linear regression. The difference is that $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ is the sigmoid output, not a raw linear value.  

---

## Coding the model

In [ ]:
class LogisticRegression:
    
    def __init__(self, learning_rate):
        self.learning_rate = learning_rate
        self.w = None
        self.b = 0.0
    
    def predict(self, X):
        z = np.dot(X, self.w) + self.b
        return sigmoid(z)
    
    def loss(self, x_train, y_train):
        m = len(x_train)
        return -(1/m)* (np.sum( (y_train*(np.log(self.predict(x_train))) + ((1-y_train)*(np.log(1-self.predict(x_train)))))))
    
    def gradient_descent(self, x_train, y_train):
        m = len(x_train)
        weight_partial = (1/m) * np.dot(x_train.T, (self.predict(x_train) - y_train))
        
        bias_partial = (1/m) * np.sum(self.predict(x_train)-y_train)
        
        self.w -= self.learning_rate * weight_partial
        self.b -= self.learning_rate * bias_partial
    
    def fit(self, x_train, y_train, epochs):
        self.w = np.zeros(x_train.shape[1])
        losses = []
        
        for i in range(epochs):
            loss = self.loss(x_train, y_train)
            self.gradient_descent(x_train, y_train)
            losses.append(loss)
        
        fig = px.line(y=losses, title="Loss vs Iteration", template="plotly_dark")
        fig.update_layout(
            title_font_color="#41BEE9",
            xaxis=dict(color="#41BEE9", title="Iterations"),
            yaxis=dict(color="#41BEE9", title="Loss")
        )

        fig.show()

model = LogisticRegression(.001)
sklearn_model = SklearnLR()

#Getting data
df = pd.read_csv('../data/student_dataset.csv')

df['placement_status'] = df['placement_status'].map({'Placed': 1, 'Not Placed': 0})

train_df = df[:8000]
test_df = df[8000:]

x_train = train_df[['study_hours', 'attendance',
                    'sleep_hours', 'internet_usage', 'assignments_completed','previous_score','exam_score']]

x_test = test_df[['study_hours', 'attendance',
                    'sleep_hours', 'internet_usage', 'assignments_completed','previous_score','exam_score']]

y_train = train_df['placement_status']
y_test = test_df['placement_status']
mean = x_train.mean()
std = x_train.std()

x_train = (x_train - mean) / std
x_test = (x_test - mean) / std

model.fit(x_train, y_train, 3500)
sklearn_model.fit(x_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

---

## Accuracy


In [ ]:
y_pred = (model.predict(x_test) >= 0.5).astype(int)

# Accuracy
accuracy = np.mean(y_pred == y_test)
 
y_pred_sklearn = sklearn_model.predict(x_test)

cm = confusion_matrix(y_test, y_pred)

fig = ff.create_annotated_heatmap(
    z=cm,
    x=['Pred 0', 'Pred 1'],
    y=['Actual 0', 'Actual 1'],
    colorscale='Blues',
    showscale=True
)

fig.update_layout(template="plotly_dark", title="Confusion Matrix")
fig.show()


cm_sklearn = confusion_matrix(y_test, y_pred_sklearn)

fig = ff.create_annotated_heatmap(
    z=cm_sklearn,
    x=['Pred 0', 'Pred 1'],
    y=['Actual 0', 'Actual 1'],
    colorscale='Blues',
    showscale=True
)

fig.update_layout(template="plotly_dark", title="Confusion Matrix - Sklearn")
fig.show()

accuracy_sklearn = np.mean(y_pred_sklearn == y_test)
print(f"Sklearn Accuracy: {accuracy_sklearn:.2%}")
print(f"Your Model Accuracy: {accuracy:.2%}")



Sklearn Accuracy: 99.65%
Your Model Accuracy: 94.85%


## Results  

My from-scratch gradient descent implementation achieved 94.85% versus sklearn's 99.65%. The gap comes from sklearn using optimized solvers like L-BFGS with regularization, while mine uses vanilla gradient descent, but the implementation correctly learns the decision boundary and validates the math.  

---

## Precision & Recall  


In [58]:
#Precision and recall
#Precision: how many students were predicted to place and how many actually were there
#Recall: Of those who placed how many were caught
 
TP = np.sum((y_pred == 1) & (y_test == 1)) #True posiitve
FP = np.sum((y_pred == 1) & (y_test == 0)) #False positive
FN = np.sum((y_pred == 0) & (y_test == 1)) #False negative

precision = TP / (TP + FP)
recall = TP / (TP + FN)

print(f"Precision: {precision:.2%}")
print(f"Recall: {recall:.2%}")

Precision: 99.87%
Recall: 94.00%


This precision percentage represents that when the model predicts a student to place it is almost always correct  
  
The recall value percentage representa that the model misses around 6% of students who actually placed